In [0]:
%run "../utils/03_write_to_delta_utils"

from pyspark.sql.functions import col, to_date, sum, avg, when


#-------------------------------------------
#    Fact Daily Sales
#-------------------------------------------

df_dim_cust = spark.read.table("novamart.gold.dim_customers")
df_dim_product = spark.read.table("novamart.gold.dim_products")
df_fact_sales = spark.read.table("novamart.gold.fact_sales")

df_joined = (
    df_fact_sales
    .join(df_dim_cust, on = "customer_id", how = "left")
    .join(df_dim_product, on = "product_id", how = "left")
)

df_agg_daily_sales = (df_joined
                      .groupBy(
                          to_date(col("transaction_timestamp")).alias("sales_date"),
                          col("category").alias("product_category"),
                          col("region").alias("customer_region")
                          )
                      .agg(
                          sum(col("total_amount")).cast("decimal(12,2)").alias("daily_sale"),
                          sum(col("quantity").cast("integer")).alias("daily_units_sold"),
                          avg(col("gross_margin_percentage").cast("decimal(12,2)")).alias("average_margin_pct")
                          )
                      )

write_delta_table(
    df = df_agg_daily_sales,
    table_name = "novamart.gold.fact_daily_sales",
    write_mode = "merge",
    merge_key = ["sales_date", "product_category", "customer_region"],
    cluster_keys = ["sales_date", "product_category", "customer_region"]
)